# Deep Ensemble — California Housing (MLP)

In [ ]:
import numpy as np
import torch, torch.nn as nn, json, os, sys
from sklearn.metrics import r2_score
sys.path.insert(0, os.path.join('.', '..'))
from shared.data_utils import load_california_housing, build_mlp_encoder

CONFIG = {'method': 'deep_ensemble', 'hidden_dims': [128, 64], 'n_models': 5,
          'epochs': 200, 'batch_size': 32, 'lr': 1e-3, 'seeds': [42, 43, 44]}
RESULT_DIR = os.path.join('.', 'results', 'deep_ensemble')
os.makedirs(RESULT_DIR, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
class EnsembleModel(nn.Module):
    def __init__(self, input_dim, hidden_dims):
        super().__init__()
        self.encoder, h_dim = build_mlp_encoder(input_dim, hidden_dims)
        self.head = nn.Linear(h_dim, 1)
    def forward(self, x): return self.head(self.encoder(x))

def train_single(seed, input_dim, train_loader, X_val, y_val):
    torch.manual_seed(seed); np.random.seed(seed)
    model = EnsembleModel(input_dim, CONFIG['hidden_dims']).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=CONFIG['lr'])
    crit = nn.MSELoss()
    best_state, best_val = None, float('inf')
    for _ in range(CONFIG['epochs']):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()
        model.eval()
        with torch.no_grad(): vl = crit(model(X_val.to(DEVICE)), y_val.to(DEVICE)).item()
        if vl < best_val: best_val = vl; best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    return best_state
print('Model defined.')

In [ ]:
def train_one_seed(seed):
    print(f'\n--- Seed {seed} ---')
    X_train, y_train, X_val, y_val, X_test, y_test, scaler, input_dim = \
        load_california_housing(random_state=seed)
    train_ds = torch.utils.data.TensorDataset(X_train, y_train)
    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True)
    all_preds = []
    for m in range(CONFIG['n_models']):
        s = seed * 100 + m
        state = train_single(s, input_dim, train_loader, X_val, y_val)
        model = EnsembleModel(input_dim, CONFIG['hidden_dims']).to(DEVICE)
        model.load_state_dict(state); model.eval()
        with torch.no_grad(): all_preds.append(model(X_test.to(DEVICE)).cpu().numpy().squeeze())
        print(f'  Model {m+1}/{CONFIG["n_models"]} done')
    all_preds = np.array(all_preds)
    return all_preds.mean(0), -all_preds.var(0), y_test.numpy().squeeze()

for seed in CONFIG['seeds']:
    y_pred, scores, y_true = train_one_seed(seed)
    sd = os.path.join(RESULT_DIR, f'seed_{seed}'); os.makedirs(sd, exist_ok=True)
    np.save(os.path.join(sd, 'test_predictions.npy'), y_pred)
    np.save(os.path.join(sd, 'test_scores.npy'), scores)
    np.save(os.path.join(sd, 'test_labels.npy'), y_true)
    print(f'  R²: {r2_score(y_true, y_pred):.4f}')
print(f'\nDone. Saved to {RESULT_DIR}')